In [1]:
import json
from pathlib import Path
from imgnet.collections.store import IndexedDatasets
from imgnet.collections.source import HuggingFaceSource
from imgnet.download import download_collection
from imgtools.nifti.crawl import Crawler
from rich import print

/home/gpudual/bhklab/josh/med-image_index/.pixi/envs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
indexed_datasets = IndexedDatasets("indexed_datasets") # This will download the latest version of the indexed_datasets if you dont provide a path
print(indexed_datasets.collections[:10])

15:07:17 [info     ] Indexed datasets path: /home/gpudual/bhklab/josh/med-image_index/notebooks/indexed_datasets [imgnet] call=store.__init__:47


[
    '4D-Lung',
    'ACRIN-6698',
    'ACRIN-Contralateral-Breast-MR',
    'ACRIN-FLT-Breast',
    'ACRIN-NSCLC-FDG-PET',
    'Adrenal-ACC-Ki67-Seg',
    'Advanced-MRI-Breast-Lesions',
    'Anti-PD-1_Lung',
    'B-mode-and-CEUS-Liver',
    'BREAST-DIAGNOSIS'
]

In [3]:
source = HuggingFaceSource(
    source="huggingface",
    repo_id="zhaodongwu/MSWAL",
    file_type="nifti",
    description="MSWAL comprises 694 high-resolution CT scans (191,417 slices) with 7 lesion types (e.g., liver tumors, kidney stones), balanced gender distribution (53.9% male, 46.1% female), and diverse CT contrast phases (non-contrast, arterial, venous, etc.), offering a fully annotated, multi-class dataset for precise abdominal lesion segmentation."
)


In [4]:
dataset_index_path = (indexed_datasets.imgtools_path / "MSWAL")
dataset_index_path.mkdir(parents=True, exist_ok=True)
with open(dataset_index_path / "source.json", "w") as f:
    json.dump(source.model_dump(mode="json"), f, indent=2)

In [5]:
print(indexed_datasets.source_config("MSWAL"))

HuggingFaceSource(
    file_type=<FileType.NIFTI: 'nifti'>,
    source='huggingface',
    repo_id='zhaodongwu/MSWAL',
    post_download=['unzip'],
    description='MSWAL comprises 694 high-resolution CT scans (191,417 slices) with 7 lesion types (e.g., liver 
tumors, kidney stones), balanced gender distribution (53.9% male, 46.1% female), and diverse CT contrast phases 
(non-contrast, arterial, venous, etc.), offering a fully annotated, multi-class dataset for precise abdominal 
lesion segmentation.'
)

In [6]:
temp_data_path = Path("temp_data") / "MSWAL"
temp_data_path.mkdir(parents=True, exist_ok=True)
download_collection(temp_data_path, source)

Fetching 970 files: 100%|██████████| 970/970 [07:15<00:00,  2.23it/s]


In [10]:
import imgtools.loggers
import logging

imgtools.loggers.get_logger("imgtools").setLevel(logging.INFO)

crawler = Crawler(
    nifti_dir=temp_data_path,
    scan_name_pattern="images{Split}/MSWAL_{ImageID:d}_0000.nii.gz",
    mask_name_pattern="labels{Split}/MSWAL_{ImageID:d}.nii.gz",
    output_dir=indexed_datasets.imgtools_path,
    n_jobs=10,
    force=True,
    deep=True,
)
crawler.crawl()

15:20:07 [info     ] Starting NIFTI crawl.          [imgtools] nifti_dir=PosixPath('temp_data/MSWAL') output_dir=PosixPath('indexed_datasets/.imgtools') dataset_name=None call=crawler.crawl:78
         [info     ] Found 968 NIfTI files in /home/gpudual/bhklab/josh/med-image_index/notebooks/temp_data/MSWAL. [imgtools] call=parse_niftis.parse_nifti_dir:426
         [info     ] Using shared keys: ['Split', 'ImageID'] for reference_id, this will be used to link masks to their referenced scans [imgtools] call=parse_niftis.parse_nifti_dir:447
15:27:54 [info     ] Parsing all NIfTI files took 467.5426 seconds [imgtools] call=timer_utils.__exit__:58
         [info     ] Saved index.                   [imgtools] path=indexed_datasets/.imgtools/MSWAL/index.csv rows=968 call=parse_niftis.parse_nifti_dir:492


In [12]:
index_df = indexed_datasets.index("MSWAL")
index_df

,Split,ImageID,filepath,file_type,reference_id,class,hash,size,ndim,nvoxels,...,mask.bbox.max_coord,mask.feret_diameter,mask.roundness,mask.flatness,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan
0,Tr,1,imagesTr/MSWAL_0001_0000.nii.gz,scan,Tr_1,MedImage,ec117e20f6d9e39b42bfcb5988e821eb1a066165,"(512, 512, 177)",3,46399488,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Tr,2,imagesTr/MSWAL_0002_0000.nii.gz,scan,Tr_2,MedImage,5041b6c35a82c0fea483e97460b690d66087ab6e,"(512, 512, 177)",3,46399488,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Tr,3,imagesTr/MSWAL_0003_0000.nii.gz,scan,Tr_3,MedImage,67cb8cf2c2c31e405099da93f5b0f4fc56856744,"(512, 512, 381)",3,99876864,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Tr,8,imagesTr/MSWAL_0008_0000.nii.gz,scan,Tr_8,MedImage,f716e03e231ad3e07b4adfc28ea701cab38842ee,"(512, 512, 201)",3,52690944,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Tr,9,imagesTr/MSWAL_0009_0000.nii.gz,scan,Tr_9,MedImage,d75e77b78dc96498200fd8c59de83d4e7c168989,"(512, 512, 177)",3,46399488,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
963,Tr,688,labelsTr/MSWAL_0688.nii.gz,mask,Tr_688,Mask,6e87ecb0b91396de2aa79ac5a2a83955f1fe7f42,"(512, 512, 474)",3,124256256,...,"(378, 346, 340)",248.788027,0.393446,2.486319,2.196475,13.487813,2286.087942,"(11.307281870773021, 28.113509098844226, 61.75...",22.0,imagesTr/MSWAL_0688_0000.nii.gz
964,Tr,690,labelsTr/MSWAL_0690.nii.gz,mask,Tr_690,Mask,b9aa4bf64397d62bffd687ebf23ac35765441b26,"(512, 512, 523)",3,137101312,...,"(344, 297, 353)",238.822051,0.685153,1.451475,1.785663,43.438991,23712.062073,"(55.860323267234016, 81.07988364160474, 144.78...",20.0,imagesTr/MSWAL_0690_0000.nii.gz
965,Tr,692,labelsTr/MSWAL_0692.nii.gz,mask,Tr_692,Mask,20ebed3cc05afe10a9e268c3ace9d45a3ae3e5c0,"(512, 512, 293)",3,76808192,...,"(379, 285, 260)",64.382698,0.480680,1.843109,1.619462,15.214846,2909.008483,"(17.2374613949725, 31.770517222565385, 51.4511...",3.0,imagesTr/MSWAL_0692_0000.nii.gz
966,Tr,693,labelsTr/MSWAL_0693.nii.gz,mask,Tr_693,Mask,c357b6330f01eadd8ceb0d4f2a33d73603169e38,"(512, 512, 240)",3,62914560,...,"(194, 306, 238)",147.898405,0.622786,4.158035,2.385840,10.546019,1397.613227,"(6.104443010089622, 25.382488557742494, 60.558...",7.0,imagesTr/MSWAL_0693_0000.nii.gz


In [24]:
def map_modality(row):
    if row["file_type"] == "scan":
        return "CT"
    elif row["file_type"] == "mask":
        return "SEG"

index_df["Modality"] = index_df.apply(map_modality, axis=1)
index_df

,Split,ImageID,filepath,file_type,reference_id,class,hash,size,ndim,nvoxels,...,mask.feret_diameter,mask.roundness,mask.flatness,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan,Modality
0,Tr,1,imagesTr/MSWAL_0001_0000.nii.gz,scan,Tr_1,MedImage,ec117e20f6d9e39b42bfcb5988e821eb1a066165,"(512, 512, 177)",3,46399488,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT
1,Tr,2,imagesTr/MSWAL_0002_0000.nii.gz,scan,Tr_2,MedImage,5041b6c35a82c0fea483e97460b690d66087ab6e,"(512, 512, 177)",3,46399488,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT
2,Tr,3,imagesTr/MSWAL_0003_0000.nii.gz,scan,Tr_3,MedImage,67cb8cf2c2c31e405099da93f5b0f4fc56856744,"(512, 512, 381)",3,99876864,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT
3,Tr,8,imagesTr/MSWAL_0008_0000.nii.gz,scan,Tr_8,MedImage,f716e03e231ad3e07b4adfc28ea701cab38842ee,"(512, 512, 201)",3,52690944,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT
4,Tr,9,imagesTr/MSWAL_0009_0000.nii.gz,scan,Tr_9,MedImage,d75e77b78dc96498200fd8c59de83d4e7c168989,"(512, 512, 177)",3,46399488,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
963,Tr,688,labelsTr/MSWAL_0688.nii.gz,mask,Tr_688,Mask,6e87ecb0b91396de2aa79ac5a2a83955f1fe7f42,"(512, 512, 474)",3,124256256,...,248.788027,0.393446,2.486319,2.196475,13.487813,2286.087942,"(11.307281870773021, 28.113509098844226, 61.75...",22.0,imagesTr/MSWAL_0688_0000.nii.gz,SEG
964,Tr,690,labelsTr/MSWAL_0690.nii.gz,mask,Tr_690,Mask,b9aa4bf64397d62bffd687ebf23ac35765441b26,"(512, 512, 523)",3,137101312,...,238.822051,0.685153,1.451475,1.785663,43.438991,23712.062073,"(55.860323267234016, 81.07988364160474, 144.78...",20.0,imagesTr/MSWAL_0690_0000.nii.gz,SEG
965,Tr,692,labelsTr/MSWAL_0692.nii.gz,mask,Tr_692,Mask,20ebed3cc05afe10a9e268c3ace9d45a3ae3e5c0,"(512, 512, 293)",3,76808192,...,64.382698,0.480680,1.843109,1.619462,15.214846,2909.008483,"(17.2374613949725, 31.770517222565385, 51.4511...",3.0,imagesTr/MSWAL_0692_0000.nii.gz,SEG
966,Tr,693,labelsTr/MSWAL_0693.nii.gz,mask,Tr_693,Mask,c357b6330f01eadd8ceb0d4f2a33d73603169e38,"(512, 512, 240)",3,62914560,...,147.898405,0.622786,4.158035,2.385840,10.546019,1397.613227,"(6.104443010089622, 25.382488557742494, 60.558...",7.0,imagesTr/MSWAL_0693_0000.nii.gz,SEG


In [25]:
index_df["BodyPartExamined"] = "Abdomen"
index_df 

,Split,ImageID,filepath,file_type,reference_id,class,hash,size,ndim,nvoxels,...,mask.roundness,mask.flatness,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan,Modality,BodyPartExamined
0,Tr,1,imagesTr/MSWAL_0001_0000.nii.gz,scan,Tr_1,MedImage,ec117e20f6d9e39b42bfcb5988e821eb1a066165,"(512, 512, 177)",3,46399488,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,Abdomen
1,Tr,2,imagesTr/MSWAL_0002_0000.nii.gz,scan,Tr_2,MedImage,5041b6c35a82c0fea483e97460b690d66087ab6e,"(512, 512, 177)",3,46399488,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,Abdomen
2,Tr,3,imagesTr/MSWAL_0003_0000.nii.gz,scan,Tr_3,MedImage,67cb8cf2c2c31e405099da93f5b0f4fc56856744,"(512, 512, 381)",3,99876864,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,Abdomen
3,Tr,8,imagesTr/MSWAL_0008_0000.nii.gz,scan,Tr_8,MedImage,f716e03e231ad3e07b4adfc28ea701cab38842ee,"(512, 512, 201)",3,52690944,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,Abdomen
4,Tr,9,imagesTr/MSWAL_0009_0000.nii.gz,scan,Tr_9,MedImage,d75e77b78dc96498200fd8c59de83d4e7c168989,"(512, 512, 177)",3,46399488,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,Abdomen
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
963,Tr,688,labelsTr/MSWAL_0688.nii.gz,mask,Tr_688,Mask,6e87ecb0b91396de2aa79ac5a2a83955f1fe7f42,"(512, 512, 474)",3,124256256,...,0.393446,2.486319,2.196475,13.487813,2286.087942,"(11.307281870773021, 28.113509098844226, 61.75...",22.0,imagesTr/MSWAL_0688_0000.nii.gz,SEG,Abdomen
964,Tr,690,labelsTr/MSWAL_0690.nii.gz,mask,Tr_690,Mask,b9aa4bf64397d62bffd687ebf23ac35765441b26,"(512, 512, 523)",3,137101312,...,0.685153,1.451475,1.785663,43.438991,23712.062073,"(55.860323267234016, 81.07988364160474, 144.78...",20.0,imagesTr/MSWAL_0690_0000.nii.gz,SEG,Abdomen
965,Tr,692,labelsTr/MSWAL_0692.nii.gz,mask,Tr_692,Mask,20ebed3cc05afe10a9e268c3ace9d45a3ae3e5c0,"(512, 512, 293)",3,76808192,...,0.480680,1.843109,1.619462,15.214846,2909.008483,"(17.2374613949725, 31.770517222565385, 51.4511...",3.0,imagesTr/MSWAL_0692_0000.nii.gz,SEG,Abdomen
966,Tr,693,labelsTr/MSWAL_0693.nii.gz,mask,Tr_693,Mask,c357b6330f01eadd8ceb0d4f2a33d73603169e38,"(512, 512, 240)",3,62914560,...,0.622786,4.158035,2.385840,10.546019,1397.613227,"(6.104443010089622, 25.382488557742494, 60.558...",7.0,imagesTr/MSWAL_0693_0000.nii.gz,SEG,Abdomen


In [14]:
import json

dataset_json = temp_data_path / "dataset.json"
with open(dataset_json, "r") as f:
    dataset = json.load(f)

dataset

{'name': 'MSWAL',
 'description': ' 3D Multi-class Segmentation of Whole Abdominal Lesions Dataset',
 'licence': 'CC BY-NC 4.0',
 'relase': 'July 8, 2025',
 'tensorImageSize': '3D',
 'file_ending': '.nii.gz',
 'channel_names': {'0': 'CT'},
 'labels': {'background': 0,
  'gallstone': 1,
  'kidney stone': 2,
  'liver tumor': 3,
  'kidney tumor': 4,
  'pancreatic cancer': 5,
  'liver cyst': 6,
  'kidney cyst': 7},
 'numTraining': 484,
 'numTest': 210,
 'training': [{'image': './imagesTr/MSWAL_0001_0000.nii.gz',
   'label': './labelsTr/MSWAL_0001.nii.gz'},
  {'image': './imagesTr/MSWAL_0002_0000.nii.gz',
   'label': './labelsTr/MSWAL_0002.nii.gz'},
  {'image': './imagesTr/MSWAL_0003_0000.nii.gz',
   'label': './labelsTr/MSWAL_0003.nii.gz'},
  {'image': './imagesTr/MSWAL_0008_0000.nii.gz',
   'label': './labelsTr/MSWAL_0008.nii.gz'},
  {'image': './imagesTr/MSWAL_0009_0000.nii.gz',
   'label': './labelsTr/MSWAL_0009.nii.gz'},
  {'image': './imagesTr/MSWAL_0011_0000.nii.gz',
   'label': './l

In [26]:
labels = dataset["labels"]
labels = ", ".join([k for k, _ in labels.items() if k != "background"])
print(labels)


gallstone, kidney stone, liver tumor, kidney tumor, pancreatic cancer, liver cyst, kidney cyst

In [28]:
index_df["ROINames"] = labels
index_df

,Split,ImageID,filepath,file_type,reference_id,class,hash,size,ndim,nvoxels,...,mask.flatness,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan,Modality,BodyPartExamined,ROINames
0,Tr,1,imagesTr/MSWAL_0001_0000.nii.gz,scan,Tr_1,MedImage,ec117e20f6d9e39b42bfcb5988e821eb1a066165,"(512, 512, 177)",3,46399488,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,Abdomen,"gallstone, kidney stone, liver tumor, kidney t..."
1,Tr,2,imagesTr/MSWAL_0002_0000.nii.gz,scan,Tr_2,MedImage,5041b6c35a82c0fea483e97460b690d66087ab6e,"(512, 512, 177)",3,46399488,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,Abdomen,"gallstone, kidney stone, liver tumor, kidney t..."
2,Tr,3,imagesTr/MSWAL_0003_0000.nii.gz,scan,Tr_3,MedImage,67cb8cf2c2c31e405099da93f5b0f4fc56856744,"(512, 512, 381)",3,99876864,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,Abdomen,"gallstone, kidney stone, liver tumor, kidney t..."
3,Tr,8,imagesTr/MSWAL_0008_0000.nii.gz,scan,Tr_8,MedImage,f716e03e231ad3e07b4adfc28ea701cab38842ee,"(512, 512, 201)",3,52690944,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,Abdomen,"gallstone, kidney stone, liver tumor, kidney t..."
4,Tr,9,imagesTr/MSWAL_0009_0000.nii.gz,scan,Tr_9,MedImage,d75e77b78dc96498200fd8c59de83d4e7c168989,"(512, 512, 177)",3,46399488,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,Abdomen,"gallstone, kidney stone, liver tumor, kidney t..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
963,Tr,688,labelsTr/MSWAL_0688.nii.gz,mask,Tr_688,Mask,6e87ecb0b91396de2aa79ac5a2a83955f1fe7f42,"(512, 512, 474)",3,124256256,...,2.486319,2.196475,13.487813,2286.087942,"(11.307281870773021, 28.113509098844226, 61.75...",22.0,imagesTr/MSWAL_0688_0000.nii.gz,SEG,Abdomen,"gallstone, kidney stone, liver tumor, kidney t..."
964,Tr,690,labelsTr/MSWAL_0690.nii.gz,mask,Tr_690,Mask,b9aa4bf64397d62bffd687ebf23ac35765441b26,"(512, 512, 523)",3,137101312,...,1.451475,1.785663,43.438991,23712.062073,"(55.860323267234016, 81.07988364160474, 144.78...",20.0,imagesTr/MSWAL_0690_0000.nii.gz,SEG,Abdomen,"gallstone, kidney stone, liver tumor, kidney t..."
965,Tr,692,labelsTr/MSWAL_0692.nii.gz,mask,Tr_692,Mask,20ebed3cc05afe10a9e268c3ace9d45a3ae3e5c0,"(512, 512, 293)",3,76808192,...,1.843109,1.619462,15.214846,2909.008483,"(17.2374613949725, 31.770517222565385, 51.4511...",3.0,imagesTr/MSWAL_0692_0000.nii.gz,SEG,Abdomen,"gallstone, kidney stone, liver tumor, kidney t..."
966,Tr,693,labelsTr/MSWAL_0693.nii.gz,mask,Tr_693,Mask,c357b6330f01eadd8ceb0d4f2a33d73603169e38,"(512, 512, 240)",3,62914560,...,4.158035,2.385840,10.546019,1397.613227,"(6.104443010089622, 25.382488557742494, 60.558...",7.0,imagesTr/MSWAL_0693_0000.nii.gz,SEG,Abdomen,"gallstone, kidney stone, liver tumor, kidney t..."


In [29]:
index_df.to_csv(indexed_datasets.imgtools_path / "MSWAL" / "index.csv", index=False)

In [1]:
from imgnet.collections.store import IndexedDatasets

indexed_datasets = IndexedDatasets()

/home/gpudual/bhklab/josh/med-image_index/.pixi/envs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
14:31:59 [info     ] Indexed datasets path: /home/gpudual/.local/share/med-imagenet/indexed_datasets [imgnet] call=store.__init__:67


In [3]:
from imgindex.model import validate_index

index = indexed_datasets.index("MSWAL")
validate_index(index, "nifti", lazy=True)

Invalid NIFTI index: {
    "SCHEMA": {
        "COLUMN_NOT_IN_DATAFRAME": [
            {
                "schema": "NiftiIndex",
                "column": "NiftiIndex",
                "check": "column_in_dataframe",
                "error": "column 'SampleID' not in dataframe. Columns in dataframe: ['Split', 'ImageID', 'filepath', 'file_type', 'reference_id', 'class', 'hash', 'size', 'ndim', 'nvoxels', 'spacing', 'origin', 'direction', 'min', 'max', 'sum', 'mean', 'std', 'variance', 'dtype_str', 'dtype_numpy', 'mask.bbox.size', 'mask.bbox.min_coord', 'mask.bbox.max_coord', 'mask.feret_diameter', 'mask.roundness', 'mask.flatness', 'mask.elongation', 'mask.equivalent_spherical_radius', 'mask.equivalent_spherical_perimeter', 'mask.equivalent_ellipsoid_diameters', 'mask.volume_count', 'reference_scan', 'Modality', 'BodyPartExamined', 'ROINames']"
            }
        ]
    }
}


In [4]:
index.rename(columns={"reference_id": "SampleID"}, inplace=True)

In [5]:
new_index = validate_index(index, "nifti", lazy=True)


In [6]:
new_index

,Split,ImageID,filepath,file_type,SampleID,class,hash,size,ndim,nvoxels,...,mask.flatness,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan,Modality,BodyPartExamined,ROINames
0,Tr,1,imagesTr/MSWAL_0001_0000.nii.gz,scan,Tr_1,MedImage,ec117e20f6d9e39b42bfcb5988e821eb1a066165,"(512, 512, 177)",3,46399488,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,Abdomen,"gallstone, kidney stone, liver tumor, kidney t..."
1,Tr,2,imagesTr/MSWAL_0002_0000.nii.gz,scan,Tr_2,MedImage,5041b6c35a82c0fea483e97460b690d66087ab6e,"(512, 512, 177)",3,46399488,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,Abdomen,"gallstone, kidney stone, liver tumor, kidney t..."
2,Tr,3,imagesTr/MSWAL_0003_0000.nii.gz,scan,Tr_3,MedImage,67cb8cf2c2c31e405099da93f5b0f4fc56856744,"(512, 512, 381)",3,99876864,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,Abdomen,"gallstone, kidney stone, liver tumor, kidney t..."
3,Tr,8,imagesTr/MSWAL_0008_0000.nii.gz,scan,Tr_8,MedImage,f716e03e231ad3e07b4adfc28ea701cab38842ee,"(512, 512, 201)",3,52690944,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,Abdomen,"gallstone, kidney stone, liver tumor, kidney t..."
4,Tr,9,imagesTr/MSWAL_0009_0000.nii.gz,scan,Tr_9,MedImage,d75e77b78dc96498200fd8c59de83d4e7c168989,"(512, 512, 177)",3,46399488,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,Abdomen,"gallstone, kidney stone, liver tumor, kidney t..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
963,Tr,688,labelsTr/MSWAL_0688.nii.gz,mask,Tr_688,Mask,6e87ecb0b91396de2aa79ac5a2a83955f1fe7f42,"(512, 512, 474)",3,124256256,...,2.486319,2.196475,13.487813,2286.087942,"(11.307281870773021, 28.113509098844226, 61.75...",22.0,imagesTr/MSWAL_0688_0000.nii.gz,SEG,Abdomen,"gallstone, kidney stone, liver tumor, kidney t..."
964,Tr,690,labelsTr/MSWAL_0690.nii.gz,mask,Tr_690,Mask,b9aa4bf64397d62bffd687ebf23ac35765441b26,"(512, 512, 523)",3,137101312,...,1.451475,1.785663,43.438991,23712.062073,"(55.860323267234016, 81.07988364160474, 144.78...",20.0,imagesTr/MSWAL_0690_0000.nii.gz,SEG,Abdomen,"gallstone, kidney stone, liver tumor, kidney t..."
965,Tr,692,labelsTr/MSWAL_0692.nii.gz,mask,Tr_692,Mask,20ebed3cc05afe10a9e268c3ace9d45a3ae3e5c0,"(512, 512, 293)",3,76808192,...,1.843109,1.619462,15.214846,2909.008483,"(17.2374613949725, 31.770517222565385, 51.4511...",3.0,imagesTr/MSWAL_0692_0000.nii.gz,SEG,Abdomen,"gallstone, kidney stone, liver tumor, kidney t..."
966,Tr,693,labelsTr/MSWAL_0693.nii.gz,mask,Tr_693,Mask,c357b6330f01eadd8ceb0d4f2a33d73603169e38,"(512, 512, 240)",3,62914560,...,4.158035,2.385840,10.546019,1397.613227,"(6.104443010089622, 25.382488557742494, 60.558...",7.0,imagesTr/MSWAL_0693_0000.nii.gz,SEG,Abdomen,"gallstone, kidney stone, liver tumor, kidney t..."


In [7]:
from pathlib import Path

new_index_dir = Path("/home/gpudual/bhklab/josh/med-image_index/new_index")
collection_path = new_index_dir / "MSWAL"
collection_path.mkdir(parents=True, exist_ok=True)

new_index.to_csv(collection_path / "index.csv", index=False)

In [4]:
import json
from pathlib import Path
source = indexed_datasets.source_config("MSWAL")
new_index_dir = Path("/home/gpudual/bhklab/josh/med-image_index/new_index")

collection_path = new_index_dir / "MSWAL"

with open(collection_path / "source.json", "w") as f:
    json.dump(source.model_dump(mode="json"), f, indent=2)

In [9]:
print([col for col in indexed_datasets.collections if indexed_datasets.file_type(col).value == "nifti"])

['AMOS', 'HCUCH-recist-dataset', 'MSWAL', 'MedicalDecathalon', 'Totalsegmentator', 'TotalsegmentatorMRI']
